In [15]:
# 1) 한 번만: 압축 풀기
!tar xzf sample_data/EnglishFnt.tgz -C sample_data/

In [16]:
# ────────────────────────────────────────────────────────────────
# 0) EnglishFnt에서 D/N/E 데이터셋 자동 구축
# ────────────────────────────────────────────────────────────────
import os
from pathlib import Path
from PIL import Image
import shutil
import random

# 원본 폰트 이미지 경로
FNT_BASE = Path("sample_data/English/Fnt")
OUT_BASE = Path("dataset")
CLASS_MAP = {'Sample014': 'D', 'Sample024': 'N', 'Sample015': 'E'}

# dataset/D, dataset/N, dataset/E 폴더 생성 및 초기화
for c in ['D', 'N', 'E']:
    d = OUT_BASE / c
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

# 클래스별로 파일 복사 (최대 2000장씩 샘플)
for sample, label in CLASS_MAP.items():
    src = FNT_BASE / sample
    dst = OUT_BASE / label
    files = list(src.glob("*.png"))
    random.shuffle(files)
    for i, f in enumerate(files[:2000]):
        img = Image.open(f).convert("L").resize((32,32))
        img.save(dst / f"{label}_{i:04d}.png")

print("EnglishFnt → dataset/D,N,E 자동 구축 완료")

# ────────────────────────────────────────────────────────────────
# 1) Keras 데이터셋 로드 및 증강
# ────────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE    = (32, 32)
BATCH_SIZE  = 64
EPOCHS      = 30
TFLITE_PATH = "dne_classifier.tflite"
CLASS_NAMES = ["D", "N", "E"]
VALID_SPLIT = 0.2
SEED        = 42

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset",
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=VALID_SPLIT,
    subset="training",
    seed=SEED
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset",
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=VALID_SPLIT,
    subset="validation",
    seed=SEED
)

normalization = layers.Rescaling(1.0 / 255)
data_augmentation = tf.keras.Sequential([
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.3),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomFlip("horizontal"),
])


def preprocess_train(x, y):
    x = tf.expand_dims(x, -1) if x.shape[-1] != 1 else x
    x = data_augmentation(x)
    x = normalization(x)
    return x, y

train_ds = train_ds.map(preprocess_train).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (normalization(x), y)).prefetch(buffer_size=tf.data.AUTOTUNE)

# ────────────────────────────────────────────────────────────────
# 2) CNN 모델 정의 및 학습
# ────────────────────────────────────────────────────────────────
def build_dne_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Conv2D(32, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.3),

        layers.Conv2D(128, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model

model = build_dne_model(IMG_SIZE + (1,), len(CLASS_NAMES))
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# ────────────────────────────────────────────────────────────────
# 3) TFLite 변환 및 저장
# ────────────────────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)
print(f"3-class TFLite model saved at '{TFLITE_PATH}'")

EnglishFnt → dataset/D,N,E 자동 구축 완료
Found 3048 files belonging to 3 classes.
Using 2439 files for training.
Found 3048 files belonging to 3 classes.
Using 609 files for validation.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             

 Total params: 157,027 (613.39 KB)

 Trainable params: 156,387 (610.89 KB)

 Non-trainable params: 640 (2.50 KB)

Epoch 1/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 17s 191ms/step - accuracy: 0.4389 - loss: 1.0589 - val_accuracy: 0.3383 - val_loss: 1.1011
Epoch 2/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7472 - loss: 0.6842 - val_accuracy: 0.3448 - val_loss: 1.3983
Epoch 3/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8292 - loss: 0.4624 - val_accuracy: 0.3448 - val_loss: 1.3917
Epoch 4/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8768 - loss: 0.3377 - val_accuracy: 0.3448 - val_loss: 1.2770
Epoch 5/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8741 - loss: 0.3568 - val_accuracy: 0.3448 - val_loss: 1.2175
Epoch 6/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9143 - loss: 0.2592 - val_accuracy: 0.5468 - val_loss: 1.0134
Epoch 7/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9087 - loss: 0.2362 - val_accuracy: 0.5895 - val_loss: 0.9644
Epoch 8/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9304 - loss: 0.2012 - val_accuracy: 0.6190 -

In [6]:
!unzip /content/row_images.zip -d table_cell/

Archive:  /content/row_images.zip
replace table_cell/row_images/images/base_001.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [10]:
import os, random, shutil

# row_images.zip을 table_cell 안에 풀어서 생긴 경로
root = "/content/table_cell/row_images"

# 실제 폴더명
img_dir = os.path.join(root, "images")
label_dir = os.path.join(root, "labels")

# train/val 폴더 생성
train_img = os.path.join(root, "images/train")
val_img   = os.path.join(root, "images/val")
train_lab = os.path.join(root, "labels/train")
val_lab   = os.path.join(root, "labels/val")

os.makedirs(train_img, exist_ok=True)
os.makedirs(val_img, exist_ok=True)
os.makedirs(train_lab, exist_ok=True)
os.makedirs(val_lab, exist_ok=True)

# 이미지 목록 불러오기
images = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".png"))]
random.shuffle(images)

# 80%/20% split
split_idx = int(len(images) * 0.8)
train_list = images[:split_idx]
val_list   = images[split_idx:]

def move_pair(img_name, dst_img_dir, dst_lab_dir):
    base, _ = os.path.splitext(img_name)
    shutil.copy2(os.path.join(img_dir, img_name), dst_img_dir)
    shutil.copy2(os.path.join(label_dir, base + ".txt"), dst_lab_dir)

for im in train_list:
    move_pair(im, train_img, train_lab)

for im in val_list:
    move_pair(im, val_img, val_lab)

print(len(train_list), "train,", len(val_list), "val")


440 train, 110 val


In [11]:
%%writefile /content/table_cell/row_images/data.yaml
path: /content/table_cell/row_images

train: images/train
val: images/val

nc: 1
names: ['cell']


Writing /content/table_cell/row_images/data.yaml


In [12]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.5 MB/s eta 0:00:00


In [13]:
from ultralytics import YOLO

# 작은 프리트레인 모델 가져오기
model = YOLO("yolov8n.pt")   # 너무 느리면 n / 더 정확히 하고 싶으면 s, m 로 바꿔도 됨

model.train(
    data="/content/table_cell/row_images/data.yaml",
    epochs=80,        # 50~100 사이 아무거나, 데이터 적으니까 너무 크게만 안 가면 됨
    imgsz=640,
    batch=8,
    patience=20,      # 개선 없으면 자동 조기 종료
    project="runs_table",
    name="cell_detector"
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/table_cell/row_images/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b8f4e8c3440>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [14]:
from ultralytics import YOLO

best = YOLO("runs_table/cell_detector/weights/best.pt")

best.predict(
    source="/content/table_cell/row_images/images/val",
    conf=0.3,
    save=True
)



image 1/110 /content/table_cell/row_images/images/val/base_003.png: 96x640 100 cells, 73.3ms
image 2/110 /content/table_cell/row_images/images/val/base_010.png: 96x640 95 cells, 11.4ms
image 3/110 /content/table_cell/row_images/images/val/base_013.png: 96x640 101 cells, 25.9ms
image 4/110 /content/table_cell/row_images/images/val/base_021.png: 96x640 101 cells, 11.7ms
image 5/110 /content/table_cell/row_images/images/val/base_023.png: 96x640 97 cells, 12.5ms
image 6/110 /content/table_cell/row_images/images/val/base_026.png: 96x640 92 cells, 11.9ms
image 7/110 /content/table_cell/row_images/images/val/base_027.png: 96x640 100 cells, 21.7ms
image 8/110 /content/table_cell/row_images/images/val/base_031.png: 96x640 91 cells, 17.3ms
image 9/110 /content/table_cell/row_images/images/val/base_034.png: 96x640 91 cells, 28.1ms
image 10/110 /content/table_cell/row_images/images/val/base_035.png: 96x640 91 cells, 28.8ms
image 11/110 /content/table_cell/row_images/images/val/base_037.png: 96x64

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'cell'}
 obb: None
 orig_img: array([[[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        ...,
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 

In [25]:
import cv2
import numpy as np
from ultralytics import YOLO
import tensorflow as tf
import os

####################################
# 1) YOLO 모델 로드
####################################

# YOLO 학습 결과(best.pt) 경로에 맞게 수정
yolo_model = YOLO("runs_table/cell_detector/weights/best.pt")


####################################
# 2) TFLite D/N/E 분류기 로드
####################################

tflite_path = "/content/dne_classifier.tflite"

if not os.path.exists(tflite_path):
    raise FileNotFoundError(f"TFLite 모델을 찾을 수 없음: {tflite_path}")

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

INPUT_H = input_details[0]['shape'][1]
INPUT_W = input_details[0]['shape'][2]
INPUT_C = input_details[0]['shape'][3]

print("TFLite input shape:", (INPUT_H, INPUT_W, INPUT_C))

# 네가 학습할 때 쓴 클래스 순서 그대로 넣기
# 예: 출력이 [p_D, p_N, p_E] 순서면 아래처럼
CLASS_NAMES = ['D', 'N', 'E']


####################################
# 3) 빈칸 판단 함수
####################################

def is_blank_cell(crop, dark_thresh=230, min_dark_ratio=0.005):
    """
    crop: (h, w, 3) BGR 이미지
    dark_thresh: 이 값보다 어두운 픽셀을 '글자'라고 봄 (0~255)
    min_dark_ratio: 어두운 픽셀 비율이 이 값보다 작으면 빈칸으로 판단
    """
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    dark_ratio = np.mean(gray < dark_thresh)
    # 어두운 픽셀 비율이 매우 작으면 사실상 빈칸
    return dark_ratio < min_dark_ratio


####################################
# 4) TFLite로 D/N/E 예측하는 함수
####################################

def predict_dne_tflite(crop_resized):
    """
    crop_resized: (H, W, C) float32 / 0~1 로 정규화된 이미지
    """
    x = np.expand_dims(crop_resized, axis=0)  # (1, H, W, C)

    # 필요시 quantization 대응 (여기선 float32 가정)
    interpreter.set_tensor(input_details[0]['index'], x)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])  # (1, num_classes)

    probs = output[0]
    pred_idx = int(np.argmax(probs))
    return CLASS_NAMES[pred_idx]


####################################
# 5) 한 줄(row) 이미지 전체 처리
####################################

def classify_row_cells(image_path, conf=0.3):
    """
    image_path: 한 줄짜리 row 이미지 경로
    return:
        labels: ['-', 'D', 'N', 'E', ...]
        boxes : [(x1,y1,x2,y2), ...]
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"이미지를 열 수 없음: {image_path}")

    # YOLO 추론
    result = yolo_model(image_path, conf=conf)[0]

    # 박스 없으면 바로 종료
    if result.boxes is None or len(result.boxes) == 0:
        print("YOLO가 아무 셀도 찾지 못함.")
        return [], []

    cells = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        cx = (x1 + x2) / 2
        cells.append({
            "cx": cx,
            "bbox": (x1, y1, x2, y2)
        })

    # 왼쪽 → 오른쪽 정렬
    cells.sort(key=lambda x: x["cx"])

    labels = []
    boxes = []

    for c in cells:
        x1, y1, x2, y2 = c["bbox"]
        crop = img[y1:y2, x1:x2]

        # 1) 빈칸인지 먼저 체크
        if is_blank_cell(crop):
            labels.append('-')
            boxes.append((x1, y1, x2, y2))
            continue

        # 2) 빈칸 아니면 TFLite D/N/E에 넣기
        resized = cv2.resize(crop, (INPUT_W, INPUT_H))

        if INPUT_C == 1:
            resized = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
            resized = resized[..., None]

        resized = resized.astype(np.float32) / 255.0

        pred_label = predict_dne_tflite(resized)
        labels.append(pred_label)
        boxes.append((x1, y1, x2, y2))

    return labels, boxes


####################################
# 6) 결과 시각화 함수
####################################

def draw_dne_on_row(image_path, labels, boxes, out_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"이미지를 열 수 없음: {image_path}")

    for (label, (x1, y1, x2, y2)) in zip(labels, boxes):
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            img,
            label,
            (x1, max(y1 - 5, 0)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

    cv2.imwrite(out_path, img)
    print("결과 저장:", out_path)


####################################
# 7) 실제 테스트 실행
####################################

# 테스트용 row 이미지로 바꿔서 실행
row_img = "/content/test4.png"

labels, boxes = classify_row_cells(row_img, conf=0.3)
print("예측 라벨:", labels)

out_file = "/content/row_dne_result.png"
draw_dne_on_row(row_img, labels, boxes, out_file)


TFLite input shape: (np.int32(32), np.int32(32), np.int32(1))

image 1/1 /content/test4.png: 96x640 125 cells, 8.8ms
Speed: 0.8ms preprocess, 8.8ms inference, 1.3ms postprocess per image at shape (1, 3, 96, 640)
예측 라벨: ['E', 'E', 'E', 'E', 'N', 'E', 'D', 'N', 'N', 'N', 'D', 'E', 'N', 'N', 'E', 'N', 'N', 'N', 'D', 'E', 'N', 'N', 'D', 'E', 'E', 'N', 'N', 'D', 'N', 'E', 'D', 'D', 'D', 'N', 'N', 'E', 'E', 'N', 'D', 'N', 'N', 'N', 'D', 'E', 'N', 'N', 'N', 'E', 'N', 'N', 'E', 'D', 'N', 'N', 'E', 'D', 'N', 'N', 'D', 'E', 'N', 'D', 'D', 'E', 'N', 'N', 'D', 'E', 'N', 'D', 'N', 'E', 'N', 'N', 'D', 'E', 'N', 'N', 'E', 'N', 'N', 'E', 'D', 'N', 'N', 'N', 'D', 'E', 'N', 'N', 'D', 'E', 'D', 'E', 'N', 'D', 'N', 'N', 'D', 'E', 'N', 'N', 'D', 'E', 'N', 'E', 'N', 'D', 'N', 'N', 'N', 'E', 'E', 'N', 'D', 'N', 'E', 'N', 'N', 'D', 'E', 'N', 'D', 'E', 'N']
결과 저장: /content/row_dne_result.png
